# Single-tool eval: `query_loki`

Runs the real production agent (`agent.agent.get_agent()`) restricted to `query_loki` **plus** its metric/label discovery helpers (`get_loki_label_values`) from `agent.tools.loki`. Everything outside this backend is still unavailable for this call — same system prompt, middleware, model, and checkpointer the Discord bot runs, just a narrower tool set. This measures how far the deployed agent gets on this backend alone.


In [ ]:
from eval_transcript import run_agent_eval
from agent.tools.loki import get_loki_label_values, query_loki


In [ ]:
TOOLS = [get_loki_label_values, query_loki]


### The loop

Every run below goes through `eval_transcript.run_agent_eval`, which drives the real production agent (`agent.agent.get_agent()`) — the same system prompt, middleware, model, and checkpointer the Discord bot runs — restricted to `TOOLS` for this call. It prints each tool call and dumps the full transcript to `eval/logs/<timestamp>__<label>.json`.


### Is the live backend up?

Both live questions below need a reachable Loki. If it isn't reachable, the runs measure error handling and nothing else — that is not a result, so skip them rather than recording a void transcript as if it were one.


In [ ]:
try:
    await get_loki_label_values.ainvoke({"label": "job"})
    LIVE_UP = True
except Exception as exc:
    LIVE_UP = False
    print(
        f"Loki unreachable ({exc}) — skipping the live cells, running the eval-dataset backend only."
    )

## In-scope question

A question the bound tools (`get_loki_label_values`, `query_loki`) together should be able to answer.

**Look for:** does it confirm the real label names before selecting on them, and does it report how much it saw?


In [ ]:
if LIVE_UP:
    answer = await run_agent_eval(
        "06_query_loki:live-inscope",
        "Show recent error logs across all services.",
        thread_id="06_query_loki:live-inscope",
        tools=TOOLS,
    )
    print(answer)


## Out-of-scope question

`query_loki` is instant-only; `query_loki_range` exists but is deliberately **not** bound here, so no bound tool can scope a 15-minute window.

**Look for:** does the model say it can't scope by time, or does it quietly answer as if it could? A refusal that blames a connection error doesn't count — that's the backend being down, not the model recognising the limit, which is why the reachability guard above matters.


In [ ]:
if LIVE_UP:
    answer = await run_agent_eval(
        "06_query_loki:live-outofscope",
        "What is the current CPU usage of the grafana container?",
        thread_id="06_query_loki:live-outofscope",
        tools=TOOLS,
    )
    print(answer)
